# 04 · Workflow steps as agentic function tools

Because every step is a single `state -> state` function, the same functions can be handed to an LLM agent as **function tools**. The agent then *drives* the bottom-up pipeline by choosing which tool to call next, while a `ToolRegistry` keeps the evolving ontology in one shared state.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make the package importable
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove for interactive plots
import bottomup_ontology as bo
print('bottomup_ontology', bo.__version__)

bottomup_ontology 0.1.0


In [2]:
corpus = [
    'A rugby player plays in a position. Siya Kolisi is a rugby player. '
    'A team has many players.',
    'A club has a coach. The coach trains the team. A player belongs to a club.',
    'Mammals such as lions and impalas live in the savanna. A lion eats impala.',
    'A blog post has a title and comments. A blog post has a category.',
]
gold_classes = {'rugby player','player','team','club','coach','lion',
                'impala','position','blog post','category','mammal'}
len(corpus)

4

## Tool specifications (OpenAI / Anthropic style JSON)
Generated automatically from the step metadata.

In [3]:
import json
specs = bo.tool_specs()
print(f'{len(specs)} tools available\n')
# show one full spec + the names of the rest
print(json.dumps(bo.tool_specs(['term_extraction'])[0], indent=2))
print()
for s in specs:
    fn = s['function']
    print(f"  {fn['name']:26s} mandatory={fn['x-mandatory']}")

7 tools available

{
  "type": "function",
  "function": {
    "name": "step_term_extraction",
    "description": "Extract candidate classes and term-composition subclass axioms. [mandatory step; techniques: term composition, formal concept analysis, hierarchical clustering, association-rule mining]. Operates on the shared workflow state and returns updated artifacts.",
    "parameters": {
      "type": "object",
      "properties": {
        "min_frequency": {
          "type": "integer",
          "minimum": 1,
          "default": 1,
          "description": "Minimum corpus frequency for a term to be kept."
        },
        "top_k": {
          "type": [
            "integer",
            "null"
          ],
          "default": null,
          "description": "Keep only the top-k highest scoring terms (null = all)."
        }
      },
      "required": []
    },
    "x-step-id": "term_extraction",
    "x-mandatory": true
  }
}

  step_text_cleaning         mandatory=True
  step_pr

## A `ToolRegistry` holds shared state and dispatches calls by name

In [4]:
from bottomup_ontology import ToolRegistry
reg = ToolRegistry(bo.WorkflowState(documents=corpus, gold_classes=gold_classes))
print('tools:', reg.tool_names())

tools: ['step_text_cleaning', 'step_preprocessing', 'step_term_extraction', 'step_relation_extraction', 'step_axiom_finding', 'step_human_in_the_loop', 'step_evaluation']


## Simulate an agent loop
A real agent would pick tools from the specs above. Here a tiny rule-based policy walks the pipeline order; each `invoke` returns a compact JSON result (the shape an agent gets back).

In [5]:
from bottomup_ontology.steps import PIPELINE_ORDER
from bottomup_ontology.tools import TOOL_NAME_TO_STEP_ID

# tool name keyed by step id, in pipeline order
step_to_tool = {sid: name for name, sid in TOOL_NAME_TO_STEP_ID.items()}
plan = [step_to_tool[sid] for sid in PIPELINE_ORDER]

for tool_name in plan:
    args = {'top_k': 8} if tool_name == 'step_term_extraction' else {}
    result = reg.invoke(tool_name, args)
    print(f"called {result['tool']:26s} -> {result['ontology']}")

called step_text_cleaning         -> {'classes': 0, 'subclass_axioms': 0, 'object_properties': 0, 'other_axioms': 0}
called step_preprocessing         -> {'classes': 0, 'subclass_axioms': 0, 'object_properties': 0, 'other_axioms': 0}
called step_term_extraction       -> {'classes': 8, 'subclass_axioms': 1, 'object_properties': 0, 'other_axioms': 0}
called step_relation_extraction   -> {'classes': 8, 'subclass_axioms': 1, 'object_properties': 5, 'other_axioms': 0}
called step_axiom_finding         -> {'classes': 11, 'subclass_axioms': 4, 'object_properties': 5, 'other_axioms': 3}
called step_human_in_the_loop     -> {'classes': 11, 'subclass_axioms': 4, 'object_properties': 5, 'other_axioms': 3}
called step_evaluation            -> {'classes': 11, 'subclass_axioms': 4, 'object_properties': 5, 'other_axioms': 3}


In [6]:
print('Final ontology built by the agent:')
print(reg.state.ontology.to_turtle())
print('Evaluation:', reg.state.evaluation.get('class_prf'))

Final ontology built by the agent:
@prefix : <http://example.org/onto#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:BlogPost a owl:Class ; rdfs:label "blog post" .
:Club a owl:Class ; rdfs:label "club" .
:Coach a owl:Class ; rdfs:label "coach" .
:Impala a owl:Class ; rdfs:label "impala" .
:Lion a owl:Class ; rdfs:label "lion" .
:LiveSavanna a owl:Class ; rdfs:label "live savanna" .
:Mammal a owl:Class ; rdfs:label "mammal" .
:Player a owl:Class ; rdfs:label "player" .
:RugbyPlayer a owl:Class ; rdfs:label "rugby player" .
:SiyaKolisi a owl:Class ; rdfs:label "siya kolisi" .
:Team a owl:Class ; rdfs:label "team" .

:Lion rdfs:subClassOf :Mammal .
:LiveSavanna rdfs:subClassOf :Mammal .
:RugbyPlayer rdfs:subClassOf :Player .
:SiyaKolisi rdfs:subClassOf :RugbyPlayer .

:has a owl:ObjectProperty ; rdfs:domain :Team ; rdfs:range :Player .
:train a owl:ObjectProperty ; 

## Direct callables
`reg.callables()` gives plain `**params -> WorkflowState` callables, handy for frameworks that bind Python functions directly.

In [7]:
calls = reg.callables()
print(list(calls)[:3], '...')

['step_text_cleaning', 'step_preprocessing', 'step_term_extraction'] ...


## Wiring into the Anthropic SDK (sketch)

The `tool_specs()` are already in the right shape. In an Anthropic tool-use loop you would:

```python
import anthropic
client = anthropic.Anthropic()
registry = ToolRegistry(WorkflowState(documents=corpus))
tools = [{'name': s['function']['name'],
          'description': s['function']['description'],
          'input_schema': s['function']['parameters']} for s in bo.tool_specs()]

# then, on each tool_use block returned by the model:
#   result = registry.invoke(block.name, block.input)
#   -> feed `result` back as a tool_result content block
```

The model decides the order; the registry guarantees the steps mutate one consistent ontology state.